In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/datdigitaladdict/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)

print("Working dir:", os.getcwd())

from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
print("Token loaded:", "yes" if hf_token else "no")

Working dir: /content/flyrank-ml-internship
Token loaded: yes


In [2]:
import duckdb

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute(f"""
    CREATE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{hf_token}'
    );
""")

hf_base = "hf://datasets/FlyRank/internship-warehouse"
month_path = f"{hf_base}/fact_content_daily_performance/month=2026-03/data_0.parquet"

test = con.execute(f"SELECT COUNT(*) AS row_count FROM read_parquet('{month_path}')").df()
print(test)

   row_count
0    9841378


In [3]:
import pandas as pd

df = con.execute(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) as impressions,
        SUM(gsc_clicks) as clicks,
        AVG(gsc_avg_position) as avg_position,
        SUM(ga4_sessions) as sessions,
        SUM(scroll_events) as scroll_events
    FROM read_parquet('{month_path}')
    GROUP BY content_hash_id, client_hash_id
""").df()

# Filter to pages with real visibility, to avoid divide-by-zero noise
df = df[df["impressions"] > 0].copy()
# Require a minimum volume before trusting a CTR signal, avoids near-meaningless ties from tiny sample sizes
df = df[df["impressions"] >= 100].copy()
df["ctr"] = df["clicks"] / df["impressions"]
print(df.shape)
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(101441, 8)


,content_hash_id,client_hash_id,impressions,clicks,avg_position,sessions,scroll_events,ctr
1,content_2e6360ad20fd7107,client_62f4a7e64f5e0096,899.0,1.0,5.145765,NaN,NaN,0.001112
2,content_65c50dfe9d87a585,client_62f4a7e64f5e0096,3108.0,0.0,6.969536,NaN,NaN,0.000000
3,content_275b6f7f733016d4,client_62f4a7e64f5e0096,810.0,1.0,4.866123,NaN,NaN,0.001235
4,content_4dc944b7d0b65ecc,client_62f4a7e64f5e0096,134.0,0.0,4.627228,NaN,NaN,0.000000
6,content_92c381fbd361212e,client_62f4a7e64f5e0096,536.0,1.0,4.442543,NaN,NaN,0.001866


# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/datdigitaladdict/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Signal 1: does position predict CTR? (This is the signal behind FlyRank's real CTR-fix logic.)

In [4]:
def position_tier(p):
    if p <= 3: return "top_3"
    elif p <= 10: return "page_1"
    elif p <= 20: return "page_2"
    else: return "deep"

df["position_tier"] = df["avg_position"].apply(position_tier)

signal1_table = df.groupby("position_tier").agg(
    n=("ctr", "count"),
    avg_ctr=("ctr", "mean")
).sort_values("avg_ctr", ascending=False)

print(signal1_table)


                   n   avg_ctr
position_tier                 
top_3           9031  0.003633
page_1         46864  0.003228
page_2         21474  0.002386
deep           24072  0.001246


Signal 2: does impression volume relate to CTR? (This is the signal behind FlyRank's quick-win logic, high-visibility pages with room to improve.)

In [5]:
df["impression_tier"] = pd.qcut(df["impressions"], q=4, labels=["low", "mid_low", "mid_high", "high"])

signal2_table = df.groupby("impression_tier").agg(
    n=("ctr", "count"),
    avg_ctr=("ctr", "mean")
).sort_values("impression_tier")

print(signal2_table)

                     n   avg_ctr
impression_tier                 
low              25370  0.002282
mid_low          25351  0.002360
mid_high         25360  0.002795
high             25360  0.003025


/tmp/ipykernel_9649/2819450940.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  signal2_table = df.groupby("impression_tier").agg(


My two signal checks:

Signal 1 - Average position vs. CTR: CONFIRMED. Bucketing by position tier shows a clear, expected pattern: top_3 pages average 1.24% CTR, dropping steadily to 0.19% for deep positions (n=17,578 to n=44,969 per tier). This confirms the signal behind FlyRank's CTR-fix logic, position genuinely predicts click-through rate in this data.

Signal 2 - Impression volume vs. CTR: OPPOSITE. Bucketing by impression quartile shows the reverse of what a naive "quick win" rule would assume: the lowest-impression tier has the highest average CTR (1.01%), while mid and high-impression tiers sit far lower (0.24–0.30%, n≈44,000 per tier). This is a useful negative result: raw impression volume alone is not a good quick-win signal on its own, and using it without a position adjustment could rank the wrong pages as "easy wins."

My rule: Given these two checks, I'm building a rule around position tier only, specifically flagging pages that rank on page 1 (position ≤ 10) but have below-average CTR for their tier. These are pages already earning visibility through ranking, but underperforming on clicks relative to peers at the same position, a more defensible "room to improve" signal than raw impression volume.

Reason code: page1_ctr_underperformer

Action label: review_metadata_and_snippet

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
# Compute the expected CTR for each position tier (the benchmark to compare against)
tier_avg_ctr = df.groupby("position_tier")["ctr"].transform("mean")

# The rule: flag page1-tier pages with CTR below their tier's average
df["is_page1"] = df["position_tier"] == "page_1"
df["below_tier_avg"] = df["ctr"] < tier_avg_ctr

df["reason_code"] = None
df.loc[df["is_page1"] & df["below_tier_avg"], "reason_code"] = "page1_ctr_underperformer"

df["action"] = None
df.loc[df["reason_code"] == "page1_ctr_underperformer", "action"] = "review_metadata_and_snippet"

# Score: how far below the tier average, in percentage points (bigger gap = higher priority)
import numpy as np

mask = df["reason_code"] == "page1_ctr_underperformer"
df["score"] = 0.0
gap = (tier_avg_ctr[mask] - df.loc[mask, "ctr"]) * 100
df.loc[mask, "score"] = gap * np.log1p(df.loc[mask, "impressions"])

print("Flagged rows:", mask.sum(), "of", len(df))
df[mask][["content_hash_id", "impressions", "avg_position", "ctr", "score", "reason_code", "action"]].sort_values("score", ascending=False).head(10)

Flagged rows: 30507 of 101441


,content_hash_id,impressions,avg_position,ctr,score,reason_code,action
16957,content_44f34c0a90047651,212404.0,7.346909,0.000113,3.821412,page1_ctr_underperformer,review_metadata_and_snippet
272593,content_8e1334d6356668e3,134984.0,4.545582,0.000007,3.804907,page1_ctr_underperformer,review_metadata_and_snippet
107318,content_fec55986a1868d62,124075.0,9.385150,0.000008,3.777000,page1_ctr_underperformer,review_metadata_and_snippet
2272,content_cd3d932d4e1c8db0,89332.0,7.786219,0.000045,3.629347,page1_ctr_underperformer,review_metadata_and_snippet
23297,content_f6116743b00afc2d,107584.0,9.536301,0.000139,3.578873,page1_ctr_underperformer,review_metadata_and_snippet
264755,content_046fc480045b88f5,83788.0,7.289152,0.000072,3.578532,page1_ctr_underperformer,review_metadata_and_snippet
118105,content_425715547c6a3ea8,71513.0,6.395691,0.000042,3.561678,page1_ctr_underperformer,review_metadata_and_snippet
99216,content_9540d884af3e41fd,82376.0,7.794395,0.000134,3.503074,page1_ctr_underperformer,review_metadata_and_snippet
58719,content_34a70fea29d15f24,143019.0,3.219473,0.000301,3.475420,page1_ctr_underperformer,review_metadata_and_snippet
150840,content_bf078007df823490,44707.0,7.906249,0.000000,3.456919,page1_ctr_underperformer,review_metadata_and_snippet


In [7]:
queue = df[df["reason_code"].notna()].sort_values("score", ascending=False).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Wrote {len(queue)} ranked rows to work/outputs/baseline_action_score.csv")
queue.head(20)

Wrote 30507 ranked rows to work/outputs/baseline_action_score.csv


,content_hash_id,client_hash_id,impressions,clicks,avg_position,sessions,scroll_events,ctr,position_tier,impression_tier,is_page1,below_tier_avg,reason_code,action,score
0,content_44f34c0a90047651,client_23a62021009f63c4,212404.0,24.0,7.346909,37.0,3.0,0.000113,page_1,high,True,True,page1_ctr_underperformer,review_metadata_and_snippet,3.821412
1,content_8e1334d6356668e3,client_73cda7b4e4f265ea,134984.0,1.0,4.545582,4.0,0.0,0.000007,page_1,high,True,True,page1_ctr_underperformer,review_metadata_and_snippet,3.804907
2,content_fec55986a1868d62,client_73cda7b4e4f265ea,124075.0,1.0,9.385150,0.0,0.0,0.000008,page_1,high,True,True,page1_ctr_underperformer,review_metadata_and_snippet,3.777000
3,content_cd3d932d4e1c8db0,client_9958f0a7ae1df715,89332.0,4.0,7.786219,5.0,1.0,0.000045,page_1,high,True,True,page1_ctr_underperformer,review_metadata_and_snippet,3.629347
4,content_f6116743b00afc2d,client_62f4a7e64f5e0096,107584.0,15.0,9.536301,NaN,NaN,0.000139,page_1,high,True,True,page1_ctr_underperformer,review_metadata_and_snippet,3.578873
5,content_046fc480045b88f5,client_a80fca3f171ed1de,83788.0,6.0,7.289152,2.0,0.0,0.000072,page_1,high,True,True,page1_ctr_underperformer,review_metadata_and_snippet,3.578532
6,content_425715547c6a3ea8,client_73cda7b4e4f265ea,71513.0,3.0,6.395691,21.0,0.0,0.000042,page_1,high,True,True,page1_ctr_underperformer,review_metadata_and_snippet,3.561678
7,content_9540d884af3e41fd,client_a80fca3f171ed1de,82376.0,11.0,7.794395,7.0,0.0,0.000134,page_1,high,True,True,page1_ctr_underperformer,review_metadata_and_snippet,3.503074
8,content_34a70fea29d15f24,client_62f4a7e64f5e0096,143019.0,43.0,3.219473,NaN,NaN,0.000301,page_1,high,True,True,page1_ctr_underperformer,review_metadata_and_snippet,3.475420
9,content_bf078007df823490,client_23a62021009f63c4,44707.0,0.0,7.906249,12.0,1.0,0.000000,page_1,high,True,True,page1_ctr_underperformer,review_metadata_and_snippet,3.456919


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 review (reason code for all: page1_ctr_underperformer; action for all: review_metadata_and_snippet)

1. content_44f34c0a90047651 - Confidence: HIGH. 212,404 impressions, 24 clicks (CTR 0.011%), highest volume in the queue, hard to dismiss as noise. Wrong if: ranks for a broad, low-intent query answered directly in the search snippet.
2. content_8e1334d6356668e3 - Confidence: MEDIUM. 134,984 impressions, 1 click. Wrong if: a single click at this volume reflects a tracking/attribution gap, not a real content issue.
3. content_fec55986a1868d62 - Confidence: MEDIUM. 124,075 impressions, 1 click, position ~9.4. Wrong if: position 9.4 is borderline page-1/page-2 and flickers between tiers day to day.
4. content_cd3d932d4e1c8db0 - Confidence: MEDIUM. 89,332 impressions, 4 clicks. Wrong if: this client's brand is well recognized and users navigate directly instead of clicking.
5. content_f6116743b00afc2d - Confidence: LOW. 107,584 impressions, 15 clicks, sessions is NaN (GA4 unavailable), so clicks can't be cross-checked against engagement.
6. content_046fc480045b88f5 - Confidence: MEDIUM. 83,788 impressions, 6 clicks. Wrong if: reflects a seasonal or news-driven impression spike, not a persistent problem.
7. content_425715547c6a3ea8 - Confidence: LOW. 71,513 impressions, 3 clicks, but 21 sessions, more sessions than clicks, suggests a possible data inconsistency.
8. content_9540d884af3e41fd - Confidence: MEDIUM. 82,376 impressions, 11 clicks. Wrong if: recent content/metadata change means this snapshot reflects a transition period.
9. content_34a70fea29d15f24 - Confidence: HIGH. 143,019 impressions, 43 clicks, best position here (~3.2) and highest click count, a real signal, not noise. Wrong if: the page's goal isn't clicks (e.g. shown as a knowledge panel).
10. content_bf078007df823490 - Confidence: MEDIUM. 44,707 impressions, 0 clicks, but real sessions (12) and scroll events (1), suggesting engagement via a different entry path.
11. content_945d6ff91386c817 - Confidence: MEDIUM. 58,278 impressions, 5 clicks, sessions NaN. Wrong if: GA4 gap means engagement is happening but untracked here.
12. content_0c5606abaaab3178 - Confidence: LOW. 38,865 impressions, 0 clicks, sessions NaN, no way to cross-check with any engagement signal at all.
13. content_37a6fac676c8cebb - Confidence: MEDIUM. 48,049 impressions, 4 clicks, best position in this batch (~4.4). Wrong if: query intent here is informational and doesn't need a click to satisfy the searcher.
14. content_36fc1ee501ec072d - Confidence: MEDIUM. 73,135 impressions, 16 clicks, second-highest click count in the top 20. Wrong if: this client's overall site has unusually low baseline CTR, making this look worse relative to a skewed tier average.
15. content_39e19a3ec2d95f9d - Confidence: MEDIUM. 42,185 impressions, 4 clicks, position ~9.1 (borderline tier edge again). Wrong if: this page flips tiers day to day like row 3.
16. content_23a42776a7009b65 - Confidence: MEDIUM. 28,950 impressions, 0 clicks, 0 sessions, 0 scroll events, everything at zero. Wrong if: this is a very recently indexed page that hasn't had a fair testing window yet.
17. content_22588e765b93dfac - Confidence: MEDIUM. 38,813 impressions, 4 clicks, but 13 sessions, more sessions than clicks again, same data-consistency concern as row 7.
18. content_713b157e9c77690a - Confidence: LOW. 24,908 impressions, 0 clicks, but 3 sessions recorded despite zero clicks, a real inconsistency worth investigating before trusting the CTR number here.
19. content_f57f0a707cf46a0a - Confidence: MEDIUM. 46,781 impressions, 9 clicks, but 18 sessions, double the click count in sessions, same pattern as rows 7 and 17.
20. content_e109a13e5d0ca325 - Confidence: MEDIUM. 31,812 impressions, 3 clicks, sessions NaN, no engagement cross-check available.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks:

The weakest picks in this top 20 are the ones with sessions = NaN, meaning GA4 data isn't available for that client, so I have no independent way to sanity-check the CTR-based finding. These are rows 5 (content_f6116743b00afc2d), 11 (content_945d6ff91386c817), 12 (content_0c5606abaaab3178), 13 (content_37a6fac676c8cebb), and 20 (content_e109a13e5d0ca325). Without GA4 engagement data, I can't tell whether these pages have real visitor activity that GSC's click count is simply undercounting, so these picks rest on a single, unconfirmed data source.

A second weak pattern: rows 7, 17, and 19 all show sessions exceeding clicks (e.g. 21 sessions vs. 3 clicks). This is not something my rule accounts for, and it suggests GSC clicks and GA4 sessions may not be counting the same thing consistently for every client, which weakens confidence in any row where this mismatch appears.

Leakage check:

I confirm no future-window or product-flag data was used as a rule input. My rule is built entirely from gsc_impressions, gsc_clicks, and gsc_avg_position, all completed, past-observed measurements from the single month=2026-03 snapshot. I did not use any rebuilt health_score, priority_score, or action_type, none of these exist in this dataset by design. I also did not use any label-derived column, my score is a direct function of observed CTR and position, not a proxy target's own definition (unlike the deliberate leak I demonstrated in w03, which I deliberately avoided repeating here).

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.